# SLURM job submission for QRW-A and QRW-M

The two hybrid walks are submitted separately:
- **QRW-M**: alternate `n_qw` quantum steps and one classical step.
- **QRW-A**: apply the convex combination `alpha * QW + (1-alpha) * RW`.


In [ ]:
import os
import json
import subprocess
import itertools
from pathlib import Path

import numpy as np


## 1. Basic settings


In [ ]:
# Run this notebook from the repository root.
PROJECT_DIR = Path.cwd()

# Create directories used by the job arrays.
(PROJECT_DIR / 'params/qrw_m').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'params/qrw_a').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'results/qrw_m').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'results/qrw_a').mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'logs').mkdir(exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")


## 2. Common utilities


In [ ]:
def generate_param_combinations(sweep_params):
    """Generate all combinations of sweep parameters."""
    param_lists = {}
    for key, value in sweep_params.items():
        if isinstance(value, list):
            param_lists[key] = value
        else:
            param_lists[key] = [value]
    
    keys = list(param_lists.keys())
    values = list(param_lists.values())
    
    combinations = []
    for combo in itertools.product(*values):
        combinations.append(dict(zip(keys, combo)))
    
    return combinations


def create_param_files(combinations, common_params, param_dir):
    """Create one JSON file for each parameter combination."""
    param_dir = Path(param_dir)
    param_dir.mkdir(parents=True, exist_ok=True)
    
    # Remove previously generated parameter files
    for f in param_dir.glob('params_*.json'):
        f.unlink()
    
    param_files = []
    for i, combo in enumerate(combinations, 1):
        params = {**common_params, **combo}
        
        filename = param_dir / f'params_{i:03d}.json'
        with open(filename, 'w') as f:
            json.dump(params, f, indent=2)
        
        param_files.append(filename)
    
    return param_files

def submit_array_job(script_path, num_jobs, time='04:00:00', exclude=None):
    script_path = Path(script_path)
    cmd = f'sbatch --array=1-{num_jobs} --time={time}'
    if exclude:
        cmd += f' --exclude={exclude}'
    cmd += f' "{script_path.name}"'
    
    print(f"Command: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                            cwd=str(script_path.parent))
    
    if result.returncode == 0:
        job_id = result.stdout.strip().split()[-1]
        print(f"✓ Submitted job array: {job_id}")
        return job_id
    else:
        print(f"✗ Submission failed: {result.stderr}")
        return None

---
# QRW-M setup
Alternate `n_qw` quantum steps and one classical step.


In [ ]:
# ==============================================
# Common QRW-M parameters
# ==============================================
QRW_M_COMMON = {
    'graph_type': 'cycle',
    'graph_size': 2501,         # 2,501 vertices and 5,002 directed arcs
    'coin_type': 'H',
    'time_length': 50000,
    'walk_type': 'QRW_M',
    'save_interval': 5,
    'output_dir': str(PROJECT_DIR / 'results/qrw_m'),
}

# ==============================================
# QRW-M parameter sweep used in the manuscript
# ==============================================
QRW_M_SWEEP = {
    'n_qw': list(range(20)),
    'n_rw': [1],
    'seed': [0],
}

# Generate parameter combinations.
qrw_m_combinations = generate_param_combinations(QRW_M_SWEEP)
print(f"QRW_M total combinations: {len(qrw_m_combinations)}")
print("\nFirst 3:")
for combination in qrw_m_combinations[:3]:
    print(f"  {combination}")


In [ ]:
# Generate QRW-M parameter files
qrw_m_param_files = create_param_files(
    qrw_m_combinations, 
    QRW_M_COMMON, 
    PROJECT_DIR / 'params/qrw_m'
)
print(f"Created {len(qrw_m_param_files)} QRW_M parameter files")

# Inspect the first file
with open(qrw_m_param_files[0]) as f:
    print("\nExample (params_001.json):")
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Submit the QRW-M job array
print(f"Ready to submit {len(qrw_m_combinations)} QRW_M jobs")

qrw_m_job_id = submit_array_job(
    PROJECT_DIR / 'submit_qrw_m.sh',
    num_jobs=len(qrw_m_combinations),
    time='96:00:00',
)

---
# QRW-A setup
Apply `alpha * QW + (1-alpha) * RW` at every step.


In [ ]:
# ==============================================
# Common QRW-A parameters
# ==============================================
QRW_A_COMMON = {
    'graph_type': 'cycle',
    'graph_size': 2501,         # 2,501 vertices and 5,002 directed arcs
    'coin_type': 'H',
    'time_length': 50000,
    'walk_type': 'QRW_A',
    'save_interval': 5,
    'output_dir': str(PROJECT_DIR / 'results/qrw_a'),
}

# ==============================================
# QRW-A parameter sweep used for the diffusive data
# ==============================================
QRW_A_SWEEP = {
    'alpha': [round(x, 2) for x in np.linspace(0.0, 0.95, 20)],
    'seed': [0],
}

qrw_a_combinations = generate_param_combinations(QRW_A_SWEEP)
print(f"QRW_A total combinations: {len(qrw_a_combinations)}")
print("\nFirst 3:")
for combination in qrw_a_combinations[:3]:
    print(f"  {combination}")


In [ ]:
# Generate QRW-A parameter files
qrw_a_param_files = create_param_files(
    qrw_a_combinations, 
    QRW_A_COMMON, 
    PROJECT_DIR / 'params/qrw_a'
)
print(f"Created {len(qrw_a_param_files)} QRW_A parameter files")

# Inspect the first file
with open(qrw_a_param_files[0]) as f:
    print("\nExample (params_001.json):")
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Submit the QRW-A job array
print(f"Ready to submit {len(qrw_a_combinations)} QRW_A jobs")

qrw_a_job_id = submit_array_job(
    PROJECT_DIR / 'submit_qrw_a.sh',
    num_jobs=len(qrw_a_combinations),
    time='7-00:00:00',
)

---
## Monitor jobs


In [ ]:
def check_job_status(job_id=None):
    """Show SLURM job status."""
    if job_id:
        cmd = ['squeue', '-j', job_id, '-o', '%.10i %.9P %.20j %.8u %.8T %.10M %.6D %R']
    else:
        cmd = ['squeue', '-u', os.environ.get('USER', ''), 
               '-o', '%.10i %.9P %.20j %.8u %.8T %.10M %.6D %R']
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)


def check_completed_results(walk_type):
    """List completed result files."""
    result_dir = PROJECT_DIR / f'results/{walk_type.lower()}'
    result_files = list(result_dir.glob('*.npz'))
    
    if walk_type == 'QRW_M':
        total = len(qrw_m_combinations)
    else:
        total = len(qrw_a_combinations)
    
    print(f"{walk_type} completed: {len(result_files)} / {total}")
    
    if result_files:
        print(f"\nLatest files:")
        for f in sorted(result_files, key=lambda x: x.stat().st_mtime)[-5:]:
            print(f"  {f.name}")
    
    return result_files

In [ ]:
# Show all jobs for the current user.
check_job_status()


In [ ]:
# Check completed QRW-M and QRW-A results.
qrw_m_results = check_completed_results('QRW_M')
qrw_a_results = check_completed_results('QRW_A')
